In [1]:
import os
import glob
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
import json

BASE_DIR = r"E:\Praxis\TERM2\DAS\retail_parquet"
MODELS_DIR = os.path.join(BASE_DIR, "mlops_pipeline", "models")
os.makedirs(MODELS_DIR, exist_ok=True)

def load_table(dir_name):
    path = os.path.join(BASE_DIR, dir_name)
    files = glob.glob(os.path.join(path, "*.parquet"))
    if not files:
        print(f"Warning: No parquet files found in {path}")
        return pd.DataFrame()
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

def prepare_data():
    print("Loading data...")
    orders = load_table("list_orders")
    order_items = load_table("list_order_items")
    customers = load_table("list_customers")
    sellers = load_table("list_sellers")
    products = load_table("list_products")
    
    # Process orders
    orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
    orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
    orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])
    
    delivered = orders[orders['order_status'] == 'delivered'].copy()
    delivered['is_late'] = (delivered['order_delivered_customer_date'] > delivered['order_estimated_delivery_date']).astype(int)
    
    # Process items and aggregate per order (simple approach: take the first item's seller/product for proxy, or sum freight)
    # A better approach for order level: sum freight and price
    order_items_agg = order_items.groupby('order_id').agg(
        revenue=('price', 'sum'),
        freight=('freight_value', 'sum'),
        items_count=('order_item_id', 'count'),
        product_id=('product_id', 'first'),
        seller_id=('seller_id', 'first')
    ).reset_index()
    
    # Merge all
    print("Merging datasets...")
    df = pd.merge(delivered[['order_id', 'customer_id', 'is_late', 'order_purchase_timestamp']], order_items_agg, on='order_id', how='inner')
    df = pd.merge(df, customers[['customer_id', 'customer_state']], on='customer_id', how='left')
    df = pd.merge(df, sellers[['seller_id', 'seller_state']], on='seller_id', how='left')
    df = pd.merge(df, products[['product_id', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']], on='product_id', how='left')
    
    # Feature Engineering
    df['product_volume'] = df['product_length_cm'] * df['product_height_cm'] * df['product_width_cm']
    df['purchase_month'] = df['order_purchase_timestamp'].dt.month
    df['purchase_dayofweek'] = df['order_purchase_timestamp'].dt.dayofweek
    
    # Select features
    features = [
        'revenue', 'freight', 'items_count', 'product_weight_g', 'product_volume',
        'customer_state', 'seller_state', 'purchase_month', 'purchase_dayofweek'
    ]
    
    X = df[features]
    y = df['is_late']
    
    return X, y

def train_model():
    X, y = prepare_data()
    print(f"Dataset shape: {X.shape}")
    
    # Handle missing values before train test split for simplicity
    numeric_features = ['revenue', 'freight', 'items_count', 'product_weight_g', 'product_volume', 'purchase_month', 'purchase_dayofweek']
    categorical_features = ['customer_state', 'seller_state']
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', Pipeline(steps=[
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]), numeric_features),
            ('cat', Pipeline(steps=[
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore'))
            ]), categorical_features)
        ])
    
    # Define pipeline
    model = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1))
    ])
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    print("Training model...")
    model.fit(X_train, y_train)
    
    print("Evaluating model...")
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    roc_auc = roc_auc_score(y_test, y_prob)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"ROC-AUC: {roc_auc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    
    # Extract Feature Importances
    # Get feature names after one-hot encoding
    cat_encoder = model.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
    cat_feature_names = cat_encoder.get_feature_names_out(categorical_features)
    all_feature_names = numeric_features + list(cat_feature_names)
    
    importances = model.named_steps['classifier'].feature_importances_
    
    feature_importance_dict = {name: float(imp) for name, imp in zip(all_feature_names, importances)}
    # Sort and take top 10
    top_features = dict(sorted(feature_importance_dict.items(), key=lambda item: item[1], reverse=True)[:10])
    
    metrics = {
        "roc_auc": float(roc_auc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "top_features": top_features
    }
    
    with open(os.path.join(MODELS_DIR, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=4)
        
    print("Saving model...")
    model_path = os.path.join(MODELS_DIR, "late_delivery_model.pkl")
    joblib.dump(model, model_path)
    print(f"Model saved to {model_path}")

if __name__ == "__main__":
    train_model()



Loading data...
Merging datasets...
Dataset shape: (96478, 9)
Training model...
Evaluating model...
ROC-AUC: 0.7070
Precision: 0.1715
Recall: 0.5291
F1 Score: 0.2590
Saving model...
Model saved to E:\Praxis\TERM2\DAS\retail_parquet\mlops_pipeline\models\late_delivery_model.pkl
